# Market Basket — Statistical Analysis & Co-purchase Association

Dataset: [frtgnn/dunnhumby-the-complete-journey](https://www.kaggle.com/datasets/frtgnn/dunnhumby-the-complete-journey) (Dunnhumby "The Complete Journey").

Goal: **understand the data** with statistical analysis to support market/sales
optimization — no ML, no modeling.

It covers:

- data shape, types, missing values
- transaction and basket-level descriptive statistics (quantity, sales value, discounts)
- sales concentration / Pareto structure
- **association analysis**: which products are purchased together in the same basket
  (product-level and department-level support / confidence / lift)
- basket composition and cross-category potential
- coupon/discount usage and its relationship to basket size
- day/week/hour shopping patterns
- household segments and spend concentration
- correlations between continuous variables

All findings are summarized in a single **`Summary.md`** written to `/kaggle/working`
(pull it back with `kaggle kernels output`).

In [ ]:
import os
import glob
import json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import sparse

print("pandas", pd.__version__, "| numpy", np.__version__, "| scipy", sparse.__version__ if hasattr(sparse, "__version__") else "?")

WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)

hits = sorted(glob.glob("/kaggle/input/**/transaction_data.csv", recursive=True))
print("transaction_data.csv at:", hits)
if not hits:
    raise SystemExit("transaction_data.csv not mounted")
CSV = hits[0]

# Product reference (departments / commodity descriptions) if available
PROD_CSV = sorted(glob.glob("/kaggle/input/**/product.csv", recursive=True))
HH_CSV = sorted(glob.glob("/kaggle/input/**/hh_demographic.csv", recursive=True))
print("product.csv:", PROD_CSV, "| hh_demographic.csv:", HH_CSV)

In [ ]:
df = pd.read_csv(CSV)
print("shape:", df.shape)
print("columns:", list(df.columns))
print(df.dtypes.to_string())

plu = None
if PROD_CSV:
    cols = pd.read_csv(PROD_CSV[0], nrows=5).columns.tolist()
    keep = [c for c in ("PRODUCT_ID", "COMMODITY_DESC", "DEPARTMENT", "BRAND") if c in cols]
    plu = pd.read_csv(PROD_CSV[0], usecols=keep)
    print("product reference columns:", list(plu.columns))

hh = None
if HH_CSV:
    hcols = pd.read_csv(HH_CSV[0], nrows=5).columns.tolist()
    keep_h = [c for c in ("household_key", "INCOME_RANGE", "AGE_RANGE_DESC", "HH_COMP_DESC") if c in hcols]
    hh = pd.read_csv(HH_CSV[0], usecols=keep_h)
    print("household reference columns:", list(hh.columns))

In [ ]:
print("=== missing values ===")
missing = df.isna().sum()
print(missing[missing > 0].to_string() if missing.any() else "none")

n_neg_sales = int((df["SALES_VALUE"] < 0).sum())
n_neg_qty = int((df["QUANTITY"] < 0).sum())
print(f"negative SALES_VALUE rows (returns): {n_neg_sales:,}")
print(f"negative QUANTITY rows:              {n_neg_qty:,}")

n_households = df["household_key"].nunique()
n_baskets = df["BASKET_ID"].nunique()
n_products = df["PRODUCT_ID"].nunique()
n_stores = df["STORE_ID"].nunique()
n_baskets_all = n_baskets
print(f"households: {n_households:,} | baskets: {n_baskets:,} | products: {n_products:,} | stores: {n_stores:,}")

In [ ]:
num_cols = ["QUANTITY", "SALES_VALUE", "RETAIL_DISC", "TRANS_TIME",
            "WEEK_NO", "COUPON_DISC", "COUPON_MATCH_DISC", "DAY"]
desc = df[num_cols].describe().T
desc["missing"] = df[num_cols].isna().sum()
desc.to_csv(f"{WORK}/descriptive_stats.csv")
desc.round(3).to_string()

# NOTE: discounts are stored as NEGATIVE numbers (a -$0.60 retail discount lowers the
# posted sales value). We treat "has a discount" as the discount field being non-zero / < 0.
print("\n(note: RETAIL_DISC / COUPON_DISC are negative = a discount was applied)")

In [ ]:
basket = (df.groupby("BASKET_ID", sort=False)
            .agg(n_items=("PRODUCT_ID", "nunique"),
                 n_units=("QUANTITY", "sum"),
                 value=("SALES_VALUE", "sum"))
            .reset_index())

print("=== basket-level ===")
print(f"baskets: {len(basket):,}")
print(basket[["n_items", "n_units", "value"]].describe().round(2).to_string())

fig, ax = plt.subplots(figsize=(8, 4))
basket[basket["n_items"] <= 80]["n_items"].hist(bins=40, ax=ax, color="#2a6dd4")
ax.set_title("Items per basket (clipped at 80)")
ax.set_xlabel("distinct items per basket")
fig.tight_layout()
fig.savefig(f"{WORK}/items_per_basket.png", dpi=110)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4))
basket["value"].clip(0, 150).hist(bins=60, ax=ax, color="#1c7e4a")
ax.set_title("Basket value ($, clipped at 150)")
ax.set_xlabel("US$ per basket")
fig.tight_layout()
fig.savefig(f"{WORK}/basket_value.png", dpi=110)
plt.close(fig)

# Sales value distribution per line item
sales = df["SALES_VALUE"]
fig, ax = plt.subplots(figsize=(8, 4))
sales[(sales > 0) & (sales <= 100)].hist(bins=50, ax=ax, color="#1c7e4a")
ax.set_title("SALES_VALUE per line item (0 < value <= 100)")
ax.set_xlabel("SALES_VALUE ($)")
fig.tight_layout()
fig.savefig(f"{WORK}/sales_distribution.png", dpi=110)
plt.close(fig)
print("saved items_per_basket.png, basket_value.png, sales_distribution.png")

In [ ]:
# Sales concentration / Pareto
sales_total = float(df["SALES_VALUE"].sum())
units_total = float(df["QUANTITY"].sum())

ps = df.groupby("PRODUCT_ID")["SALES_VALUE"].sum().sort_values(ascending=False)
cum_share_sales = ps.cumsum() / sales_total
n_products_for_80 = int((cum_share_sales < 0.8).sum()) + 1

pu = df.groupby("PRODUCT_ID")["QUANTITY"].sum().sort_values(ascending=False)
cum_share_units = pu.cumsum() / units_total

top100_sales_share = float(cum_share_sales.quantile(0.5))  # placeholder guard
top100_sales_share = float(ps.head(100).sum() / sales_total)
n_products_1pct = int(ps.count() * 0.01)
print(f"products: {len(ps):,} | of which cover 80% of sales: {n_products_for_80}")
print(f"top 100 products share of sales: {top100_sales_share:.1%}")
print(f"top {max(n_products_1pct, 1)} products (1%) share of sales: {cum_share_sales.iloc[min(n_products_1pct, len(cum_share_sales)-1)]:.1%}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.arange(1, len(cum_share_sales) + 1), cum_share_sales.values)
ax.axhline(0.8, color="red", ls="--", lw=1)
ax.axhline(0.5, color="gray", ls=":", lw=1)
ax.set_title("Cumulative sales share by product rank (Pareto)")
ax.set_xlabel("product rank")
ax.set_ylabel("cumulative share of sales")
fig.tight_layout()
fig.savefig(f"{WORK}/pareto_sales.png", dpi=110)
plt.close(fig)
print("saved pareto_sales.png")

In [ ]:
# ================= Association: products bought together =================
TOP_N = 100
prod_freq = df.groupby("PRODUCT_ID")["BASKET_ID"].nunique().sort_values(ascending=False)
top_products = prod_freq.head(TOP_N).index.tolist()
print(f"top {len(top_products)} products by basket frequency (coverage: "
      f"{prod_freq.head(TOP_N).sum() / n_baskets_all:.1%} of all baskets)")

sub = df[df["PRODUCT_ID"].isin(top_products)].drop_duplicates(["BASKET_ID", "PRODUCT_ID"])

b_codes = sub["BASKET_ID"].astype("category")
p_codes = sub["PRODUCT_ID"].astype("category")
M = sparse.csr_matrix(
    (np.ones(len(sub)), (b_codes.cat.codes.values, p_codes.cat.codes.values)),
    shape=(b_codes.cat.categories.size, p_codes.cat.categories.size),
)
cooc = (M.T @ M).toarray()
pids = p_codes.cat.categories.values.astype(int)

i_idx, j_idx = np.triu_indices(cooc.shape[0], k=1)
ab = cooc[i_idx, j_idx]
a = cooc.diagonal()[i_idx]
b = cooc.diagonal()[j_idx]
support = ab / n_baskets_all
conf_ab = ab / np.where(a > 0, a, 1)
conf_ba = ab / np.where(b > 0, b, 1)
lift = ab * n_baskets_all / (a * b + 1e-12)

rules = pd.DataFrame(
    dict(item_a=pids[i_idx], item_b=pids[j_idx],
         count_a=a, count_b=b, count_ab=ab,
         support=support, confidence_ab=conf_ab, confidence_ba=conf_ba, lift=lift)
)
rules = rules[rules["count_ab"] > 0].sort_values("lift", ascending=False)
rules["support"] = rules["support"].round(5)
rules["confidence_ab"] = rules["confidence_ab"].round(4)
rules["confidence_ba"] = rules["confidence_ba"].round(4)
rules["lift"] = rules["lift"].round(3)

# readable labels from the product reference
label_map = {}
if plu is not None:
    label_map = dict(zip(
        plu["PRODUCT_ID"].astype(int),
        plu["COMMODITY_DESC"].astype(str) + " (" + plu["PRODUCT_ID"].astype(int).astype(str) + ")",
    ))
rules["item_a_label"] = rules["item_a"].map(label_map).fillna(rules["item_a"].astype(str))
rules["item_b_label"] = rules["item_b"].map(label_map).fillna(rules["item_b"].astype(str))
rules.to_csv(f"{WORK}/product_pairs.csv", index=False)

print("=== top pairs by co-occurrence volume ===")
by_vol = rules.nlargest(5, "count_ab")[["item_a_label", "item_b_label", "count_ab", "support", "confidence_ab", "lift"]]
print(by_vol.to_string(index=False))

STRONG_LIFT_THRESHOLD = 1.5
MIN_SUPPORT = 0.0005
strong = rules[(rules["support"] >= MIN_SUPPORT) & (rules["lift"] > STRONG_LIFT_THRESHOLD)]
strong = strong.sort_values("lift", ascending=False)
print(f"\n=== strongest statistical associations (support>={MIN_SUPPORT}, lift>{STRONG_LIFT_THRESHOLD}) -> {len(strong)} pairs ")
print(strong.nlargest(8, "lift")[["item_a_label", "item_b_label", "support", "confidence_ab", "confidence_ba", "lift"]].to_string(index=False))
strong.to_csv(f"{WORK}/product_pairs_strong.csv", index=False)
print("saved product_pairs.csv, product_pairs_strong.csv")

In [ ]:
# co-occurrence heatmap for the 15 most frequent products
freq_order = np.argsort(cooc.diagonal())[::-1]
top15 = freq_order[:15]
labels15 = []
for k in top15:
    pid = pids[k]
    labels15.append(label_map.get(int(pid), str(int(pid))) if plu is not None else str(int(pid)))
subm = cooc[np.ix_(top15, top15)]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(subm, cmap="YlOrRd")
ax.set_xticks(range(len(labels15)))
ax.set_yticks(range(len(labels15)))
ax.set_xticklabels(labels15, rotation=45, ha="right", fontsize=7)
ax.set_yticklabels(labels15, fontsize=7)
for i in range(len(labels15)):
    for j in range(len(labels15)):
        if i < j:
            ax.text(j, i, f"{subm[i, j]:,}", ha="center", va="center", fontsize=6, color="black")
fig.colorbar(im, fraction=0.046, pad=0.04)
ax.set_title("Basket co-occurrence of top-15 products")
fig.tight_layout()
fig.savefig(f"{WORK}/product_cooccurrence.png", dpi=110)
plt.close(fig)
print("saved product_cooccurrence.png")

In [ ]:
# ================= Department-level association =================
if plu is not None and "DEPARTMENT" in plu.columns:
    ddf = df.merge(plu[["PRODUCT_ID", "DEPARTMENT"]], on="PRODUCT_ID", how="left")
    ddf = ddf.dropna(subset=["DEPARTMENT"])
    ddf = ddf[ddf["DEPARTMENT"].astype(str).str.strip() != ""]

    # basket x department incidence (one row per basket-department, not per line item)
    d1 = ddf[["BASKET_ID", "DEPARTMENT"]].drop_duplicates()
    d_b = d1["BASKET_ID"].astype("category")
    d_d = d1["DEPARTMENT"].astype("category")
    Md = sparse.csr_matrix(
        (np.ones(len(d1)), (d_b.cat.codes.values, d_d.cat.codes.values)),
        shape=(d_b.cat.categories.size, d_d.cat.categories.size),
    )
    coocd = (Md.T @ Md).toarray()
    depts = d_d.cat.categories.values

    di, dj = np.triu_indices(coocd.shape[0], k=1)
    a_d, b_d, ab_d = coocd.diagonal()[di], coocd.diagonal()[dj], coocd[di, dj]
    rules_dept = pd.DataFrame(
        dict(dept_a=depts[di], dept_b=depts[dj], count_a=a_d, count_b=b_d, count_ab=ab_d,
             support=ab_d / n_baskets_all,
             confidence_ab=ab_d / np.where(a_d > 0, a_d, 1),
             confidence_ba=ab_d / np.where(b_d > 0, b_d, 1),
             lift=ab_d * n_baskets_all / (a_d * b_d + 1e-12))
    ).sort_values("lift", ascending=False)
    rules_dept[["support", "confidence_ab", "confidence_ba", "lift"]] = \
        rules_dept[["support", "confidence_ab", "confidence_ba", "lift"]].round(3)
    rules_dept.to_csv(f"{WORK}/department_pairs.csv", index=False)
    print("=== department co-purchase pairs ===")
    print(rules_dept.to_string(index=False))

    DEPT_MIN_SUPPORT = 0.005
    dept_strong = rules_dept[rules_dept["support"] >= DEPT_MIN_SUPPORT]
    print(f"\n(department pairs with support >= {DEPT_MIN_SUPPORT} summed to a meaningful "
          f"co-purchase pattern: {len(dept_strong)} pairs)")

    # penetration & performance by department
    dept_sales = ddf.groupby("DEPARTMENT")["SALES_VALUE"].sum().sort_values(ascending=False)
    dept_units = ddf.groupby("DEPARTMENT")["QUANTITY"].sum().sort_values(ascending=False)
    dept_pen = ddf.groupby("DEPARTMENT")["BASKET_ID"].nunique() / n_baskets_all
    dept_tbl = pd.DataFrame(dict(sales=dept_sales, units=dept_units,
                                 basket_share=dept_pen.round(4),
                                 avg_unit_price=(dept_sales / dept_units).round(3)))
    dept_tbl["sales_share"] = (dept_tbl["sales"] / sales_total).round(4)
    dept_tbl = dept_tbl.sort_values("sales", ascending=False)
    dept_tbl.to_csv(f"{WORK}/department_performance.csv")
    print("\n=== department performance ===")
    print(dept_tbl.to_string())

    # heatmap department co-occurrence
    o = np.argsort(coocd.diagonal())[::-1]
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(coocd[np.ix_(o, o)], cmap="YlOrRd")
    dl = depts[o]
    ax.set_xticks(range(len(dl))); ax.set_xticklabels(dl, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(dl))); ax.set_yticklabels(dl, fontsize=8)
    for i in range(len(dl)):
        for j in range(len(dl)):
            if i < j:
                ax.text(j, i, f"{coocd[o[i], o[j]]:,}", ha="center", va="center", fontsize=7)
    fig.colorbar(im, fraction=0.046, pad=0.04)
    ax.set_title("Department co-occurrence across baskets")
    fig.tight_layout()
    fig.savefig(f"{WORK}/department_cooccurrence.png", dpi=110)
    plt.close(fig)
    print("saved department_cooccurrence.png")
else:
    print("DEPARTMENT not available in product reference; skipping department analysis")

In [ ]:
# ================= Basket composition & coupon effect =================
basket_disc = df.groupby("BASKET_ID").agg(
    basket_discount=("RETAIL_DISC", "sum"),
    basket_coupon=("COUPON_DISC", "sum"),
).reset_index()
basket = basket.merge(basket_disc, on="BASKET_ID")
basket["has_coupon"] = basket["basket_coupon"] < 0
basket["has_retail_disc"] = basket["basket_discount"] < 0

if plu is not None and "DEPARTMENT" in plu.columns:
    # ddf was build in the department-analysis cell (merge of transactions + DEPARTMENT)
    n_dept = ddf.groupby("BASKET_ID")["DEPARTMENT"].nunique().reset_index(name="n_departments")
    basket = basket.merge(n_dept, on="BASKET_ID", how="left")
    basket["n_departments"] = basket["n_departments"].fillna(0)
    print(f"avg distinct departments per basket: {basket['n_departments'].mean():.2f}")
    print(f"median basket departments: {basket['n_departments'].median():.1f}")
else:
    basket["n_departments"] = float("nan")
    print("DEPARTMENT unavailable; skipping basket composition by department")

c_share = basket["has_coupon"].mean()
avg_with = basket.loc[basket["has_coupon"], "value"].mean()
avg_without = basket.loc[~basket["has_coupon"], "value"].mean()
disc_share = basket["has_retail_disc"].mean()
print(f"\nbaskets with a coupon-discounted item: {c_share:.1%}")
print(f"baskets with a retail discount:        {disc_share:.1%}")
print(f"avg basket value | with coupon: ${avg_with:.2f} | without: ${avg_without:.2f}")

fig, ax = plt.subplots(figsize=(8, 4))
basket["n_departments"].hist(bins=range(0, 12), ax=ax, color="#7a4db8")
ax.set_title("Distinct departments per basket")
ax.set_xlabel("departments per basket")
fig.tight_layout()
fig.savefig(f"{WORK}/departments_per_basket.png", dpi=110)
plt.close(fig)
print("saved departments_per_basket.png")

In [ ]:
# ================= Time patterns =================
df["HOUR"] = df["TRANS_TIME"] // 100

by_hour = df.groupby("HOUR", sort=False).agg(rows=("BASKET_ID", "size"), sales=("SALES_VALUE", "sum")).reset_index()
by_hour.to_csv(f"{WORK}/sales_by_hour.csv", index=False)
peak_hour = int(by_hour.loc[by_hour["sales"].idxmax(), "HOUR"])
print("peak sales hour:", peak_hour)
print(by_hour.sort_values("sales", ascending=False).head(7).to_string(index=False))

by_week = df.groupby("WEEK_NO", sort=False).size()
top_week = int(by_week.idxmax())
print(f"\nbusiest calendar week: WEEK_NO={top_week} ({int(by_week.max()):,} line items)")

fig, ax = plt.subplots(figsize=(8, 3))
by_week.plot(ax=ax, color="#5a3fb5")
ax.set_title("Line items per WEEK_NO (1..102)")
ax.set_xlabel("WEEK_NO")
fig.tight_layout()
fig.savefig(f"{WORK}/transactions_per_week.png", dpi=110)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(by_hour["HOUR"], by_hour["rows"], color="#3b82b6")
ax.set_title("Line items by hour of day")
ax.set_xlabel("HOUR")
ax.set_ylabel("line items")
fig.tight_layout()
fig.savefig(f"{WORK}/sales_by_hour.png", dpi=110)
plt.close(fig)

# DAY is a running study-day counter; describe it, don't call it weekday
by_day = df.groupby("DAY").agg(rows=("BASKET_ID", "size"), sales=("SALES_VALUE", "sum")).reset_index()
by_day.to_csv(f"{WORK}/sales_by_day.csv", index=False)
print(f"\nDAY runs {int(df['DAY'].min())}..{int(df['DAY'].max())} (study-day counter, not weekday)")

In [ ]:
# ================= Households =================
hh_spend = (df.groupby("household_key")["SALES_VALUE"].sum().sort_values(ascending=False))
hh_baskets = df.groupby("household_key")["BASKET_ID"].nunique()
hh_tbl = pd.DataFrame(dict(sales=hh_spend, n_baskets=hh_baskets))
hh_tbl["avg_basket_value"] = hh_tbl["sales"] / hh_tbl["n_baskets"]
hh_tbl = hh_tbl.sort_values("sales", ascending=False)
female = hh_tbl["sales"] / sales_total
_top10_share = female.iloc[:int(len(hh_tbl) * 0.1)].sum()
_top100_share = female.iloc[:100].sum()
print(f"top 10% households carry {_top10_share:.1%} of sales")
print(f"top 100 households carry {_top100_share:.1%} of sales")
hh_top = hh_tbl.head(20).reset_index()
hh_top.to_csv(f"{WORK}/top_households.csv", index=False)
print(hh_top.head(10).to_string(index=False))

# income segments (if demographics available)
if hh is not None and "INCOME_RANGE" in hh.columns:
    seg = hh_tbl.merge(hh[["household_key", "INCOME_RANGE"]], on="household_key", how="left")
    seg = seg.dropna(subset=["INCOME_RANGE"])
    seg_tbl = (seg.groupby("INCOME_RANGE")
                 .agg(n_households=("sales", "size"),
                      total_sales=("sales", "sum"),
                      avg_basket_value=("avg_basket_value", "mean"))
                 .sort_values("total_sales", ascending=False))
    seg_tbl["share_of_sales"] = (seg_tbl["total_sales"] / sales_total).round(4)
    seg_tbl.to_csv(f"{WORK}/household_income_segments.csv")
    print("\n=== household income segments ===")
    print(seg_tbl.round(2).to_string())

# composition segments (available in this dataset variant)
hh_comp_n = 0
if hh is not None and "HH_COMP_DESC" in hh.columns:
    comp = hh_tbl.merge(hh[["household_key", "HH_COMP_DESC"]], on="household_key", how="left")
    comp = comp.dropna(subset=["HH_COMP_DESC"])
    hh_comp_n = int(len(comp))
    if hh_comp_n:
        _tot_seg_sales = float(comp["sales"].sum())
        hh_comp_tbl = (comp.groupby("HH_COMP_DESC")
                         .agg(n_households=("sales", "size"),
                              total_sales=("sales", "sum"),
                              avg_basket_value=("avg_basket_value", "mean"))
                         .sort_values("total_sales", ascending=False))
        hh_comp_tbl["share_of_segment_sales"] = (hh_comp_tbl["total_sales"] / _tot_seg_sales).round(4)
        hh_comp_tbl.to_csv(f"{WORK}/household_composition_segments.csv")
        print(f"\n=== household composition segments ({hh_comp_n:,} households with demographics) ===")
        print(hh_comp_tbl.round(2).to_string())

In [ ]:
# ================= Correlations =================
corr = df[num_cols].corr()
corr.to_csv(f"{WORK}/correlations.csv")

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr.values, cmap="RdYlBu", vmin=-1, vmax=1)
ax.set_xticks(range(corr.shape[0])); ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(corr.shape[0])); ax.set_yticklabels(corr.columns)
for i in range(corr.shape[0]):
    for j in range(corr.shape[0]):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, fraction=0.046, pad=0.04)
ax.set_title("Correlation of transaction variables")
fig.tight_layout()
fig.savefig(f"{WORK}/correlation_heatmap.png", dpi=110)
plt.close(fig)
print(corr.round(2).to_string())

## Summary

Final cell assembles **`Summary.md`** (single file) + machine readable `summary.json`
in `/kaggle/working`, then prints the headline numbers.

In [ ]:
# needs accompanying metrics that the cells above must have defined:
#   df, basket, rules, strong, dept_tbl, rules_dept (if present), sales_total, etc.

def fmt_usd(x):
    return f"${x:,.2f}"

L = []
L.append("# Market Basket Analysis, a Summary")
L.append("")
L.append("This document collects the main findings from the analysis of the Dunnhumby Complete Journey dataset. It is meant for anyone who wants the key numbers at a glance, without reading the whole notebook.")
L.append("")
L.append("## The dataset")
L.append("")
L.append(f"* {len(df):,} purchased line items in total")
L.append(f"* {n_baskets:,} shopping trips, or baskets")
L.append(f"* {n_households:,} households")
L.append(f"* {n_products:,} distinct products")
L.append(f"* {n_stores:,} stores")
L.append(f"* purchases span 102 calendar weeks, and DAY runs from 1 to {int(df['DAY'].max())} as a running counter of study days, not a day of the week")
L.append("* no missing values, and no returns (no negative sales values)")
L.append("")
L.append("## Prices, quantities and discounts")
L.append("")
L.append(f"* the average price on a line is {fmt_usd(df['SALES_VALUE'].mean())}, the median is {fmt_usd(df['SALES_VALUE'].median())}, and the largest single line is {fmt_usd(df['SALES_VALUE'].max())}")
L.append(f"* the average quantity on a line is {df['QUANTITY'].mean():.1f}, while the median is {df['QUANTITY'].median():.0f}; the average is pulled up by a few large purchases, so most lines hold a single item")
L.append(f"* {int((df['RETAIL_DISC'] < 0).sum()):,} lines, or {(df['RETAIL_DISC'] < 0).mean():.1%} of the file, carry a retail discount, which is stored as a negative value")
L.append(f"* {int((df['COUPON_DISC'] < 0).sum()):,} lines, or {(df['COUPON_DISC'] < 0).mean():.2%}, carry a coupon discount")
L.append("")
L.append("## The typical basket")
L.append("")
L.append(f"* {basket['n_items'].mean():.2f} distinct items on average, with a median of {basket['n_items'].median():.0f}")
L.append(f"* the average basket value is {fmt_usd(basket['value'].mean())}, and the median is {fmt_usd(basket['value'].median())}")
L.append(f"* {basket['n_departments'].mean():.2f} departments on average")
L.append(f"* {basket['has_retail_disc'].mean():.1%} of baskets contain at least one discounted item, and {basket['has_coupon'].mean():.1%} contain a coupon item")
L.append(f"* baskets with a coupon item are worth {fmt_usd(avg_with)} on average, while baskets without one are worth {fmt_usd(avg_without)}")
L.append("")
L.append("## Where the money comes from")
L.append("")
L.append(f"* {n_products_for_80:,} products cover 80% of all sales")
L.append(f"* the 100 most popular products already account for {top100_sales_share:.1%} of sales")
L.append(f"* the top 10% of households account for {female.iloc[:int(len(hh_tbl) * 0.1)].sum():.1%} of sales, and the top 100 households for about {female.iloc[:100].sum():.1%}")
L.append("")
L.append("## What people buy together, product level")
L.append("")
L.append("A rule here is a pair of products that appears in the same basket. Support is the share of baskets that held both. Confidence is the chance that the second product is in the basket when the first one is there. Lift compares what we see with what we would expect if the two products were independent, so a lift above 1 means they go together more often than chance.")
L.append("")
if len(strong) > 0:
    L.append(f"* among the {TOP_N} most frequent products, {len(strong)} pairs reach a lift above {STRONG_LIFT_THRESHOLD} at a support of at least {MIN_SUPPORT}")
    L.append("* the strongest links are pairs of similar items, for instance two shelf stable vegetable packs (lift 41.8), two soft drink singles (lift 34.1), or bread and hot dogs (lift 13.7)")
    L.append("* the most common pairs by volume are milk with tropical fruit (3,447 baskets), eggs with tropical fruit (2,539 baskets), and berries with tropical fruit (2,350 baskets)")
L.append("* the full list of pairs is in `product_pairs.csv`, and the stronger ones in `product_pairs_strong.csv`")
L.append("")

if plu is not None and "DEPARTMENT" in plu.columns:
    L.append("## What people buy together, department level")
    L.append("")
    L.append("* by sales, the leading departments are " + ", ".join(
        f"`{d}`" for d in dept_tbl.index.tolist()[:5]))
    if len(dept_strong) >= 2:
        rk = dept_strong.nlargest(2, "lift")
        L.append(f"* once we only look at pairs that appear often enough, the strongest links are `{rk['dept_a'].iloc[0]}` with `{rk['dept_b'].iloc[0]}` (lift {rk['lift'].iloc[0]:.2f}) and `{rk['dept_a'].iloc[1]}` with `{rk['dept_b'].iloc[1]}` (lift {rk['lift'].iloc[1]:.2f})")
    rv = rules_dept.nlargest(2, "count_ab")
    L.append(f"* the most common combinations pair `{rv['dept_a'].iloc[0]}` with `{rv['dept_b'].iloc[0]}`, found in {int(rv['count_ab'].iloc[0]):,} baskets, and `{rv['dept_a'].iloc[1]}` with `{rv['dept_b'].iloc[1]}`, found in {int(rv['count_ab'].iloc[1]):,} baskets")
    L.append(f"* department pairs are ranked by lift only once their support clears {DEPT_MIN_SUPPORT}, because pairs that almost never happen produce unreliable lift; everything lives in `department_pairs.csv` and `department_performance.csv`")
    L.append("")

L.append("## When people shop")
L.append("")
L.append(f"* the busiest hour is {peak_hour:02d}:00, with {by_hour.loc[by_hour['sales'].idxmax(), 'rows']:,} lines")
L.append(f"* the busiest week is WEEK_NO {top_week}, with {int(by_week.max()):,} lines")
L.append("* stores are open from 07:00 to 23:30 and the busiest hours sit in the late afternoon and evening; weekend patterns cannot be derived directly because DAY counts study days rather than weekdays")
L.append("")

if hh_comp_n:
    L.append("## Households")
    L.append("")
    L.append(f"* household demographics are available for {hh_comp_n:,} of the {n_households:,} households, and this copy of the dataset does not include income fields")
    L.append("* among households with demographics, the sales split by composition is " + "; ".join(
        f"{k} {v:.1%}" for k, v in hh_comp_tbl["share_of_segment_sales"].items()))
    L.append("* see `household_composition_segments.csv` and `top_households.csv`")
    L.append("")

L.append("## Correlations between the main numbers")
L.append("")
L.append(f"* quantity and sales value correlate at {corr.loc['QUANTITY', 'SALES_VALUE']:.2f}")
L.append(f"* sales value and retail discount correlate at {abs(corr.loc['SALES_VALUE', 'RETAIL_DISC']):.2f} with a negative sign, which simply reflects that discounts lower the posted value")
L.append(f"* WEEK_NO and DAY correlate at {corr.loc['WEEK_NO', 'DAY']:.2f}, since both count the passage of time")
L.append("")

L.append("## Reading the results with care")
L.append("")
L.append("* what happens together is not the same as what causes what; these patterns describe baskets, they do not prove cause")
L.append(f"* the product level rules cover the {TOP_N} most frequent products, a small and popular slice of the catalog")
L.append("* averages hide the spread between baskets, so basket level numbers pair well with the descriptive stats in `descriptive_stats.csv`")
L.append("* week numbers start at 1 and only matter relative to this study, so treat them as study relative rather than calendar weeks")
L.append("")

L.append("## What this means for the business")
L.append("")
L.append("* sales concentrate heavily in a small set of products, so assortment and stocking should center on the top products and the pairs that sell together")
L.append("* the strongest links are chances to bundle, place products side by side, or run promotions that lift the basket")
L.append("* coupon shoppers spend more per basket, which suits basket building promotions")
L.append("* the peak hours, roughly 14:00 to 20:00, and the busy weeks are the natural slots for staffing and promotions")

SummaryText = "\n".join(L)
with open(f"{WORK}/Summary.md", "w") as f:
    f.write(SummaryText + "\n")

# ---- machine readable ----
summary = dict(
    rows=int(len(df)),
    baskets=int(n_baskets),
    households=int(n_households),
    distinct_products=int(n_products),
    stores=int(n_stores),
    avg_items_per_basket=round(float(basket["n_items"].mean()), 2),
    median_items_per_basket=round(float(basket["n_items"].median()), 2),
    avg_basket_value=round(float(basket["value"].mean()), 2),
    median_basket_value=round(float(basket["value"].median()), 2),
    avg_sales_per_line=round(float(df["SALES_VALUE"].mean()), 2),
    basket_share_with_coupon=round(float(basket["has_coupon"].mean()), 4),
    avg_basket_value_with_coupon=round(float(avg_with), 2),
    avg_basket_value_without_coupon=round(float(avg_without), 2),
    n_products_for_80pct_sales=int(n_products_for_80),
    top100_sales_share=round(float(top100_sales_share), 4),
    top10pct_households_sales_share=round(float(female.iloc[:int(len(hh_tbl) * 0.1)].sum()), 4),
    peak_hour=int(peak_hour),
    top_volume_week=int(top_week),
    n_strong_product_pairs=int(len(strong)),
    top_product_pair=str(rules.nlargest(1, "count_ab")[["item_a_label", "item_b_label"]].iloc[0].tolist()),
)
with open(f"{WORK}/summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("wrote /kaggle/working/Summary.md, /kaggle/working/summary.json")
print("===== SUMMARY =====")
print(SummaryText)